# NLP Interface PoC — Natural Language → malariagen_data API

This notebook demonstrates how plain-language questions from researchers can be translated into executable `malariagen_data` API calls using rule-based NLP (intent classification + entity extraction).

**Example:**
> *"Show me allele frequencies for Vgsc in Kenya"*
> → `ag3.plot_frequencies_heatmap(transcript="AGAP004707-RD", sample_query="country == 'Kenya'", min_cohort_size=10)`

In [1]:
# Cell 1: NLP Interface PoC — Natural Language → malariagen_data API
#
# Goal: Demonstrate how plain-language questions from researchers
# can be translated into executable malariagen_data API calls.
#
# This PoC uses rule-based NLP (intent + entity extraction) to show
# the concept. A full implementation would use fine-tuned LLMs.

import re
from dataclasses import dataclass, field
from typing import  Dict, Any

print("NLP Interface PoC — Natural Language → malariagen_data API")

NLP Interface PoC — Natural Language → malariagen_data API


## 1. API Method Registry

Registry of 7 `malariagen_data` API methods with their parameters, types, and keyword lists. In a production system, this would be auto-generated from `describe_api()` (PR #904) and type introspection.

In [2]:
# API Method Registry — mirrors describe_api() output + parameter-level
# detail that an NLP system needs to generate valid calls.

API_METHODS = {
    "plot_frequencies_heatmap": {
        "description": "Plot a heatmap of allele frequencies for SNPs in a gene transcript",
        "category": "plot",
        "parameters": {
            "transcript": {"type": "str", "description": "Gene transcript ID (e.g., AGAP004707-RD for Vgsc)"},
            "sample_sets": {"type": "str|list", "description": "Sample set(s) to include", "optional": True},
            "sample_query": {"type": "str", "description": "Pandas query to filter samples", "optional": True},
            "area": {"type": "Region", "description": "Geographic area to filter by"},
            "min_cohort_size": {"type": "int", "description": "Minimum samples per cohort", "default": 10},
        },
        "keywords": ["frequency", "frequencies", "allele", "heatmap", "snp", "mutation", "variant", "resistance"],
    },
    "plot_frequencies_time_series": {
        "description": "Plot allele frequencies over time for SNPs in a gene",
        "category": "plot",
        "parameters": {
            "transcript": {"type": "str", "description": "Gene transcript ID"},
            "area": {"type": "Region", "description": "Geographic area"},
            "period_by": {"type": "str", "description": "Time period grouping (year, quarter, month)", "default": "year"},
        },
        "keywords": ["time", "trend", "temporal", "over time", "change", "year", "series"],
    },
    "pairwise_average_fst": {
        "description": "Compute pairwise average Fst between cohorts",
        "category": "analysis",
        "parameters": {
            "contig": {"type": "str", "description": "Chromosome/contig name (e.g., 2L, 3R, X)"},
            "cohorts": {"type": "str|dict", "description": "Cohort column or dict mapping names to queries"},
            "sample_sets": {"type": "str|list", "description": "Sample set(s)", "optional": True},
            "min_cohort_size": {"type": "int", "description": "Minimum samples per cohort", "default": 10},
        },
        "keywords": ["fst", "differentiation", "genetic distance", "population structure"],
    },
    "plot_pairwise_average_fst": {
        "description": "Plot a heatmap of pairwise Fst values between cohorts",
        "category": "plot",
        "parameters": {
            "fst_df": {"type": "DataFrame", "description": "Output from pairwise_average_fst()"},
            "annotation": {"type": "str", "description": "Upper triangle content: None, 'standard error', 'Z score', 'lower triangle'", "optional": True},
        },
        "keywords": ["fst", "heatmap", "pairwise", "plot", "visualize", "compare populations"],
    },
    "plot_snps_dxy": {
        "description": "Plot absolute genetic divergence (Dxy) across a contig",
        "category": "plot",
        "parameters": {
            "contig": {"type": "str", "description": "Chromosome/contig"},
            "cohort1_query": {"type": "str", "description": "Query for first cohort"},
            "cohort2_query": {"type": "str", "description": "Query for second cohort"},
            "window_size": {"type": "int", "description": "Window size in base pairs", "default": 20000},
        },
        "keywords": ["dxy", "divergence", "absolute", "between populations", "between species"],
    },
    "sample_metadata": {
        "description": "Access sample metadata including collection location, date, species, and partner",
        "category": "data",
        "parameters": {
            "sample_sets": {"type": "str|list", "description": "Sample set(s) to include", "optional": True},
            "sample_query": {"type": "str", "description": "Pandas query to filter", "optional": True},
        },
        "keywords": ["metadata", "samples", "sample", "collection", "location", "country", "species", "info", "list", "have", "available"],
    },
    "snp_calls": {
        "description": "Access SNP genotype calls for a genomic region",
        "category": "data",
        "parameters": {
            "region": {"type": "str", "description": "Genomic region (contig or contig:start-end)"},
            "sample_sets": {"type": "str|list", "description": "Sample set(s)", "optional": True},
            "sample_query": {"type": "str", "description": "Pandas query to filter", "optional": True},
        },
        "keywords": ["snp", "genotype", "calls", "variants", "data", "raw"],
    },
}

print(f"Registered {len(API_METHODS)} API methods:")
for name, info in API_METHODS.items():
    print(f"  [{info['category']:>8}] {name}: {info['description'][:60]}...")

Registered 7 API methods:
  [    plot] plot_frequencies_heatmap: Plot a heatmap of allele frequencies for SNPs in a gene tran...
  [    plot] plot_frequencies_time_series: Plot allele frequencies over time for SNPs in a gene...
  [analysis] pairwise_average_fst: Compute pairwise average Fst between cohorts...
  [    plot] plot_pairwise_average_fst: Plot a heatmap of pairwise Fst values between cohorts...
  [    plot] plot_snps_dxy: Plot absolute genetic divergence (Dxy) across a contig...
  [    data] sample_metadata: Access sample metadata including collection location, date, ...
  [    data] snp_calls: Access SNP genotype calls for a genomic region...


## 2. Entity Knowledge Base

Domain-specific dictionaries mapping natural language terms (gene names, countries, chromosomes, species) to valid API parameter values.

In [3]:
# Entity Dictionaries — map natural language terms to valid API parameter values.

# Gene name → transcript ID mapping
GENE_TRANSCRIPTS = {
    "vgsc": "AGAP004707-RD",
    "voltage-gated sodium channel": "AGAP004707-RD",
    "kdr": "AGAP004707-RD",
    "knockdown resistance": "AGAP004707-RD",
    "rdl": "AGAP006028-RA",
    "ace1": "AGAP001356-RA",
    "acetylcholinesterase": "AGAP001356-RA",
    "cyp6aa1": "AGAP002862-RA",
    "cyp6p3": "AGAP002865-RB",
    "cyp9k1": "AGAP000818-RA",
    "gste2": "AGAP009194-RA",
}

# Country → ISO code / area mapping
COUNTRY_AREAS = {
    "kenya": "KE",
    "tanzania": "TZ",
    "uganda": "UG",
    "ghana": "GH",
    "burkina faso": "BF",
    "cameroon": "CM",
    "mali": "ML",
    "mozambique": "MZ",
    "democratic republic of congo": "CD",
    "drc": "CD",
    "ethiopia": "ET",
    "nigeria": "NG",
    "senegal": "SN",
    "guinea": "GN",
    "ivory coast": "CI",
    "cote d'ivoire": "CI",
    "angola": "AO",
    "malawi": "MW",
    "zambia": "ZM",
    "benin": "BJ",
    "gabon": "GA",
    "gambia": "GM",
    "east africa": ["KE", "TZ", "UG", "ET", "MZ", "MW"],
    "west africa": ["GH", "BF", "ML", "NG", "SN", "GN", "CI", "BJ", "GM", "GA"],
}

# Chromosome/contig aliases
CONTIG_ALIASES = {
    "chromosome 2l": "2L",
    "chromosome 2r": "2R",
    "chromosome 3l": "3L",
    "chromosome 3r": "3R",
    "chromosome x": "X",
    "chr 2l": "2L", "chr 2r": "2R", "chr 3l": "3L", "chr 3r": "3R", "chr x": "X",
    "2l": "2L", "2r": "2R", "3l": "3L", "3r": "3R", "x": "X",
}

# Species name normalization
SPECIES_MAP = {
    "gambiae": "gambiae",
    "coluzzii": "coluzzii",
    "arabiensis": "arabiensis",
    "an. gambiae": "gambiae",
    "an. coluzzii": "coluzzii",
    "an. arabiensis": "arabiensis",
    "anopheles gambiae": "gambiae",
    "anopheles coluzzii": "coluzzii",
}

# Reverse lookup: area code → country name (for region expansion)
AREA_TO_COUNTRY = {}
for country, code in COUNTRY_AREAS.items():
    if isinstance(code, str) and country not in ("east africa", "west africa"):
        AREA_TO_COUNTRY[code] = country.title()

print(f"Knowledge base loaded:")
print(f"  {len(GENE_TRANSCRIPTS)} gene/transcript mappings")
print(f"  {len(COUNTRY_AREAS)} country/region mappings")
print(f"  {len(CONTIG_ALIASES)} contig aliases")
print(f"  {len(SPECIES_MAP)} species mappings")

Knowledge base loaded:
  11 gene/transcript mappings
  24 country/region mappings
  15 contig aliases
  8 species mappings


## 3. NLP Engine

Three-stage pipeline:
1. **Intent Classification** — keyword scoring to match query → API method, with category and context boosts
2. **Entity Extraction** — pattern matching to extract genes, countries, contigs, species (supports multi-entity extraction)
3. **API Call Generation** — template-based code generation with region expansion and multi-species handling

In [4]:
# NLP Engine - Intent Classification + Entity Extraction

@dataclass
class ParsedQuery:
    """Structured representation of a parsed natural language query."""
    raw_query: str
    intent: str  # The API method to call
    confidence: float  # 0-1 confidence score
    entities: Dict[str, Any] = field(default_factory=dict)
    api_call: str = ""  # Generated Python code
    explanation: str = ""  # Human-readable explanation

def classify_intent(query: str) -> tuple:
    """Match a query to the most relevant API method using keyword scoring."""
    query_lower = query.lower()
    scores = {}
    
    for method_name, method_info in API_METHODS.items():
        score = 0
        keywords = method_info["keywords"]
        
        for keyword in keywords:
            if keyword in query_lower:
                # Longer keyword matches get higher weight
                score += len(keyword.split())
        
        # Boost for exact method category hints
        if "plot" in query_lower or "show" in query_lower or "visualize" in query_lower:
            if method_info["category"] == "plot":
                score += 1
        if "compute" in query_lower or "calculate" in query_lower:
            if method_info["category"] == "analysis":
                score += 1
        if "get" in query_lower or "access" in query_lower or "list" in query_lower:
            if method_info["category"] == "data":
                score += 1
        # Boost "what ... samples/have" queries for data methods
        if "what" in query_lower and ("have" in query_lower or "available" in query_lower or "samples" in query_lower):
            if method_info["category"] == "data":
                score += 2
                
        # Context-specific boosts
        if "over time" in query_lower or "trend" in query_lower:
            if method_name == "plot_frequencies_time_series":
                score += 3
        if "compare" in query_lower and "population" in query_lower:
            if "fst" in method_name:
                score += 2
        if "heatmap" in query_lower and "fst" in query_lower:
            if method_name == "plot_pairwise_average_fst":
                score += 3
        
        # Multi-species divergence queries → plot_snps_dxy
        if ("between" in query_lower and ("species" in query_lower or "population" in query_lower)):
            if "divergence" in query_lower or "dxy" in query_lower:
                if method_name == "plot_snps_dxy":
                    score += 4
                
        scores[method_name] = score
    
    if not scores or max(scores.values()) == 0:
        return "unknown", 0.0
    
    best_method = max(scores, key=scores.get)
    max_score = scores[best_method]
    # Normalize confidence (cap at 1.0)
    confidence = min(max_score / 5.0, 1.0)
    
    return best_method, confidence

def extract_entities(query: str) -> Dict[str, Any]:
    """Extract structured entities from a natural language query."""
    query_lower = query.lower()
    entities = {}
    
    # Extract gene/transcript
    for gene_name, transcript_id in GENE_TRANSCRIPTS.items():
        if gene_name in query_lower:
            entities["gene"] = gene_name
            entities["transcript"] = transcript_id
            break
    
    # Extract country/area — check longest names first, use word boundaries
    for country_name, area_code in sorted(COUNTRY_AREAS.items(), key=lambda x: len(x[0]), reverse=True):
        pattern = r'\b' + re.escape(country_name) + r'\b'
        if re.search(pattern, query_lower):
            entities["country"] = country_name
            entities["area"] = area_code
            break
    
    # Extract contig/chromosome
    for alias, contig in CONTIG_ALIASES.items():
        if alias in query_lower:
            entities["contig"] = contig
            break
    
    # Extract ALL species mentioned (not just the first) — handles
    # multi-species queries like "divergence between gambiae and coluzzii"
    found_species = []
    for species_name, normalized in sorted(SPECIES_MAP.items(), key=lambda x: len(x[0]), reverse=True):
        if species_name in query_lower:
            if normalized not in [s[1] for s in found_species]:
                found_species.append((species_name, normalized))
    
    if len(found_species) == 1:
        entities["species"] = found_species[0][1]
    elif len(found_species) >= 2:
        entities["species_1"] = found_species[0][1]
        entities["species_2"] = found_species[1][1]
        entities["multi_species"] = True
    
    # Extract cohort-related hints
    if "by country" in query_lower:
        entities["cohort_by"] = "country"
    elif "by species" in query_lower or "by taxon" in query_lower:
        entities["cohort_by"] = "taxon"
    elif "by region" in query_lower:
        entities["cohort_by"] = "admin1_iso"
    
    # Extract annotation preferences
    if "lower triangle" in query_lower:
        entities["annotation"] = "lower triangle"
    elif "standard error" in query_lower or "with errors" in query_lower:
        entities["annotation"] = "standard error"
    elif "z score" in query_lower or "z-score" in query_lower:
        entities["annotation"] = "Z score"
    
    return entities

def generate_api_call(method: str, entities: Dict[str, Any]) -> tuple:
    """Generate executable Python code from method + entities."""
    
    if method == "plot_frequencies_heatmap":
        transcript = entities.get("transcript", "AGAP004707-RD")
        gene = entities.get("gene", "Vgsc")
        area = entities.get("area", None)
        
        params = [f'transcript="{transcript}"']
        if area:
            if isinstance(area, list):
                # Region list → expand to individual country filters
                country_names = [AREA_TO_COUNTRY.get(c, c) for c in area]
                filter_expr = " or ".join([f"country == '{c}'" for c in country_names])
                params.append(f'sample_query="{filter_expr}"')
            else:
                params.append(f'sample_query="country == \'{entities.get("country", "").title()}\'"')
        params.append('min_cohort_size=10')
        
        code = f"ag3.plot_frequencies_heatmap(\n    {(','+chr(10)+'    ').join(params)}\n)"
        explanation = f"Plots allele frequency heatmap for {gene} gene"
        if entities.get("country"):
            explanation += f" in {entities['country'].title()}"
            
    elif method == "plot_frequencies_time_series":
        transcript = entities.get("transcript", "AGAP004707-RD")
        gene = entities.get("gene", "Vgsc")
        area = entities.get("area", None)
        
        params = [f'transcript="{transcript}"']
        if area and not isinstance(area, list):
            params.append(f'area=Region("{area}")')
        params.append('period_by="year"')
        params.append('min_cohort_size=10')
        
        code = f"ag3.plot_frequencies_time_series(\n    {(','+chr(10)+'    ').join(params)}\n)"
        explanation = f"Plots temporal trends of {gene} mutation frequencies"
        if entities.get("country"):
            explanation += f" in {entities['country'].title()}"
            
    elif method == "pairwise_average_fst":
        contig = entities.get("contig", "3L")
        cohort_by = entities.get("cohort_by", "country")
        
        params = [f'contig="{contig}"', f'cohorts="{cohort_by}"', 'min_cohort_size=10']
        code = f"fst_df = ag3.pairwise_average_fst(\n    {(','+chr(10)+'    ').join(params)}\n)"
        explanation = f"Computes pairwise Fst on chromosome {contig} grouped by {cohort_by}"

    elif method == "plot_pairwise_average_fst":
        annotation = entities.get("annotation", None)
        params = ['fst_df']
        if annotation:
            params.append(f'annotation="{annotation}"')
        
        code = f"ag3.plot_pairwise_average_fst(\n    {(','+chr(10)+'    ').join(params)}\n)"
        explanation = f"Plots Fst heatmap"
        if annotation:
            explanation += f" with {annotation} display"

    elif method == "plot_snps_dxy":
        contig = entities.get("contig", "2L")
        params = [f'contig="{contig}"']
        
        # Handle multi-species queries with separate cohort queries
        if entities.get("multi_species"):
            params.append(f'cohort1_query="taxon == \'{entities["species_1"]}\'"')
            params.append(f'cohort2_query="taxon == \'{entities["species_2"]}\'"')
        elif entities.get("species"):
            params.append(f'cohort1_query="taxon == \'{entities["species"]}\'"')
        params.append('window_size=20000')
        
        code = f"ag3.plot_snps_dxy(\n    {(','+chr(10)+'    ').join(params)}\n)"
        explanation = f"Plots Dxy across chromosome {contig}"
        if entities.get("multi_species"):
            explanation += f" between {entities['species_1']} and {entities['species_2']}"

    elif method == "sample_metadata":
        query_parts = []
        if entities.get("species"):
            query_parts.append(f"taxon == '{entities['species']}'")
        elif entities.get("species_1"):
            query_parts.append(f"taxon == '{entities['species_1']}'")
        
        # Region expansion — "East Africa" → individual country filters
        area = entities.get("area")
        if isinstance(area, list):
            country_names = [AREA_TO_COUNTRY.get(c, c) for c in area]
            country_filter = "country in " + str(country_names)
            query_parts.append(country_filter)
        elif entities.get("country"):
            query_parts.append(f"country == '{entities['country'].title()}'")
        
        params = []
        if query_parts:
            params.append(f'sample_query="{" and ".join(query_parts)}"')
        
        code = f"ag3.sample_metadata(\n    {(','+chr(10)+'    ').join(params) if params else ''}\n)"
        explanation = "Retrieves sample metadata"
        if entities.get("species"):
            explanation += f" for {entities['species']}"
        elif entities.get("species_1"):
            explanation += f" for {entities['species_1']}"
        if entities.get("country"):
            explanation += f" in {entities['country'].title()}"

    elif method == "snp_calls":
        region = entities.get("transcript", entities.get("contig", "2L"))
        params = [f'region="{region}"']
        if entities.get("species"):
            params.append(f'sample_query="taxon == \'{entities["species"]}\'"')
        elif entities.get("country"):
            params.append(f'sample_query="country == \'{entities["country"].title()}\'"')
        
        code = f"ag3.snp_calls(\n    {(','+chr(10)+'    ').join(params)}\n)"
        explanation = f"Retrieves SNP genotype data for {entities.get('gene', region)}"
            
    else:
        code = "# Could not generate API call — query not understood"
        explanation = "Query could not be mapped to an API method"
    
    return code, explanation

def parse_query(query: str) -> ParsedQuery:
    """Full NLP pipeline: query → intent → entities → API call."""
    intent, confidence = classify_intent(query)
    entities = extract_entities(query)
    api_call, explanation = generate_api_call(intent, entities)
    
    return ParsedQuery(
        raw_query=query,
        intent=intent,
        confidence=confidence,
        entities=entities,
        api_call=api_call,
        explanation=explanation,
    )

print("NLP engine loaded")
print("Components: intent classifier, entity extractor, API call generator")

NLP engine loaded
Components: intent classifier, entity extractor, API call generator


## 4. Demo - Query Translation

Ten natural-language queries translated to executable `malariagen_data` API calls, plus three edge-case queries that fall outside the system's scope.

In [5]:
# Demo - Natural Language → API Translation

TEST_QUERIES = [
    "Show me allele frequencies for Vgsc in Kenya",
    "How have kdr mutation frequencies changed over time in Ghana?",
    "Compare genetic differentiation between populations by country on chromosome 3L",
    "Plot the Fst heatmap with lower triangle only",
    "What samples do we have from Tanzania?",
    "Get SNP genotype data for Ace1 in Uganda",
    "Show me the frequency trends for cyp6p3 in Burkina Faso",
    "What is the genetic divergence between gambiae and coluzzii populations?",
    "List all samples of An. arabiensis from East Africa",
    "Visualize insecticide resistance mutations in Mozambique",
]

print("=" * 70)
print("  NATURAL LANGUAGE → malariagen_data API TRANSLATION DEMO")
print("=" * 70)

for i, query in enumerate(TEST_QUERIES, 1):
    result = parse_query(query)
    
    print(f"\n{'─' * 70}")
    print(f'  Query {i}: "{result.raw_query}"')
    print(f"{'─' * 70}")
    print(f"  Intent:     {result.intent} (confidence: {result.confidence:.0%})")
    print(f"  Entities:   {result.entities}")
    print(f"  Explanation: {result.explanation}")
    print(f"\n  Generated API call:")
    for line in result.api_call.split('\n'):
        print(f"    {line}")

print(f"\n{'=' * 70}")
print(f"  Processed {len(TEST_QUERIES)} queries successfully")
print(f"{'=' * 70}")

  NATURAL LANGUAGE → malariagen_data API TRANSLATION DEMO

──────────────────────────────────────────────────────────────────────
  Query 1: "Show me allele frequencies for Vgsc in Kenya"
──────────────────────────────────────────────────────────────────────
  Intent:     plot_frequencies_heatmap (confidence: 60%)
  Entities:   {'gene': 'vgsc', 'transcript': 'AGAP004707-RD', 'country': 'kenya', 'area': 'KE'}
  Explanation: Plots allele frequency heatmap for vgsc gene in Kenya

  Generated API call:
    ag3.plot_frequencies_heatmap(
        transcript="AGAP004707-RD",
        sample_query="country == 'Kenya'",
        min_cohort_size=10
    )

──────────────────────────────────────────────────────────────────────
  Query 2: "How have kdr mutation frequencies changed over time in Ghana?"
──────────────────────────────────────────────────────────────────────
  Intent:     plot_frequencies_time_series (confidence: 100%)
  Entities:   {'gene': 'kdr', 'transcript': 'AGAP004707-RD', 'country'

## 5. Edge Cases — Queries Outside System Scope

Three queries that fall outside the system's scope, demonstrating graceful degradation and clarification prompts.

In [6]:
# Edge Cases — queries the system cannot handle

EDGE_CASE_QUERIES = [
    "What is the weather like in Nairobi?",
    "Tell me about malaria treatment drugs",
    "Run a GWAS analysis on chromosome 2L with PCA",
]

print("=" * 70)
print("  EDGE CASES — Queries Outside System Scope")
print("=" * 70)

for i, query in enumerate(EDGE_CASE_QUERIES, 1):
    result = parse_query(query)
    
    print(f"\n{'─' * 70}")
    print(f'  Edge Case {i}: "{result.raw_query}"')
    print(f"{'─' * 70}")
    print(f"  Intent:     {result.intent} (confidence: {result.confidence:.0%})")
    print(f"  Entities:   {result.entities}")
    print(f"  Explanation: {result.explanation}")
    if result.confidence < 0.3:
        print(f"  ⚠ LOW CONFIDENCE — system would ask for clarification")

print(f"\n{'=' * 70}")
print(f"  All {len(EDGE_CASE_QUERIES)} out-of-scope queries handled gracefully")
print(f"{'=' * 70}")

  EDGE CASES — Queries Outside System Scope

──────────────────────────────────────────────────────────────────────
  Edge Case 1: "What is the weather like in Nairobi?"
──────────────────────────────────────────────────────────────────────
  Intent:     unknown (confidence: 0%)
  Entities:   {}
  Explanation: Query could not be mapped to an API method
  ⚠ LOW CONFIDENCE — system would ask for clarification

──────────────────────────────────────────────────────────────────────
  Edge Case 2: "Tell me about malaria treatment drugs"
──────────────────────────────────────────────────────────────────────
  Intent:     unknown (confidence: 0%)
  Entities:   {}
  Explanation: Query could not be mapped to an API method
  ⚠ LOW CONFIDENCE — system would ask for clarification

──────────────────────────────────────────────────────────────────────
  Edge Case 3: "Run a GWAS analysis on chromosome 2L with PCA"
──────────────────────────────────────────────────────────────────────
  Intent:     u

## 6. Pipeline Trace — Single Query Deep Dive

Traces a single query through all three pipeline stages, showing the intermediate keyword scores, entity extraction results, and final code generation. This is the Query 8 fix — the original PoC extracted only one species and routed to the wrong method.

In [7]:
# Pipeline Trace — show all intermediate steps for a single query

trace_query = "What is the genetic divergence between gambiae and coluzzii populations?"
print("=" * 70)
print("  PIPELINE TRACE — Single Query Deep Dive")
print("=" * 70)
print(f'\n  Input: "{trace_query}"')

# Stage 1: Intent Classification — show keyword scores for all methods
print(f"\n  ── Stage 1: Intent Classification ──")
query_lower = trace_query.lower()
for method_name, method_info in API_METHODS.items():
    score = 0
    matched_keywords = []
    for keyword in method_info["keywords"]:
        if keyword in query_lower:
            score += len(keyword.split())
            matched_keywords.append(keyword)
    
    # Category boosts
    if "plot" in query_lower or "show" in query_lower or "visualize" in query_lower:
        if method_info["category"] == "plot":
            score += 1
    if "compute" in query_lower or "calculate" in query_lower:
        if method_info["category"] == "analysis":
            score += 1
    if "get" in query_lower or "access" in query_lower or "list" in query_lower:
        if method_info["category"] == "data":
            score += 1
    if "what" in query_lower and ("have" in query_lower or "available" in query_lower or "samples" in query_lower):
        if method_info["category"] == "data":
            score += 2
    # Context boosts
    if ("between" in query_lower and ("species" in query_lower or "population" in query_lower)):
        if "divergence" in query_lower or "dxy" in query_lower:
            if method_name == "plot_snps_dxy":
                score += 4
                matched_keywords.append("+divergence_between_boost")
    
    if score > 0:
        print(f"    {method_name:40s} score={score:2d}  matched: {matched_keywords}")

intent, confidence = classify_intent(trace_query)
print(f"\n  → Selected: {intent} (confidence: {confidence:.0%})")

# Stage 2: Entity Extraction
print(f"\n  ── Stage 2: Entity Extraction ──")
entities = extract_entities(trace_query)
for key, value in entities.items():
    print(f"    {key:20s} → {value}")

# Stage 3: API Call Generation
print(f"\n  ── Stage 3: API Call Generation ──")
api_call, explanation = generate_api_call(intent, entities)
print(f"    Method: ag3.{intent}()")
print(f"    Explanation: {explanation}")
print(f"\n    Generated code:")
for line in api_call.split('\n'):
    print(f"      {line}")

  PIPELINE TRACE — Single Query Deep Dive

  Input: "What is the genetic divergence between gambiae and coluzzii populations?"

  ── Stage 1: Intent Classification ──
    plot_snps_dxy                            score= 5  matched: ['divergence', '+divergence_between_boost']

  → Selected: plot_snps_dxy (confidence: 100%)

  ── Stage 2: Entity Extraction ──
    species_1            → coluzzii
    species_2            → gambiae
    multi_species        → True

  ── Stage 3: API Call Generation ──
    Method: ag3.plot_snps_dxy()
    Explanation: Plots Dxy across chromosome 2L between coluzzii and gambiae

    Generated code:
      ag3.plot_snps_dxy(
          contig="2L",
          cohort1_query="taxon == 'coluzzii'",
          cohort2_query="taxon == 'gambiae'",
          window_size=20000
      )


## 7. Results Summary

Accuracy analysis across all 10 test queries.

In [8]:
# Results Summary - Accuracy Analysis

results = []
for query in TEST_QUERIES:
    r = parse_query(query)
    results.append(r)

# Print summary table
print(f"{'Query':<55} {'Intent':<35} {'Conf':>5} {'Entities':>3}")
print("─" * 100)
for r in results:
    entity_count = len(r.entities)
    short_query = r.raw_query[:52] + "..." if len(r.raw_query) > 55 else r.raw_query
    print(f"{short_query:<55} {r.intent:<35} {r.confidence:>4.0%} {entity_count:>3}")

# Statistics
avg_confidence = sum(r.confidence for r in results) / len(results)
intents_found = sum(1 for r in results if r.intent != "unknown")
total_entities = sum(len(r.entities) for r in results)

print(f"\n{'─' * 100}")
print(f"Average confidence: {avg_confidence:.0%}")
print(f"Queries resolved:   {intents_found}/{len(results)}")
print(f"Total entities extracted: {total_entities}")

Query                                                   Intent                               Conf Entities
────────────────────────────────────────────────────────────────────────────────────────────────────
Show me allele frequencies for Vgsc in Kenya            plot_frequencies_heatmap             60%   4
How have kdr mutation frequencies changed over time ... plot_frequencies_time_series        100%   4
Compare genetic differentiation between populations ... pairwise_average_fst                 60%   2
Plot the Fst heatmap with lower triangle only           plot_pairwise_average_fst           100%   1
What samples do we have from Tanzania?                  sample_metadata                     100%   2
Get SNP genotype data for Ace1 in Uganda                snp_calls                            80%   4
Show me the frequency trends for cyp6p3 in Burkina Faso plot_frequencies_time_series        100%   4
What is the genetic divergence between gambiae and c... plot_snps_dxy                

## 8. Interactive Interface

Try your own queries against the NLP pipeline. Uncomment `interactive_demo()` to run interactively.

In [9]:
# Interactive Query Interface

def interactive_demo():
    """Run an interactive query session."""
    print("\nMalariaGEN NLP Interface — Interactive Demo")
    print("Type a question about mosquito genomic data.")
    print("Type 'quit' to exit.\n")
    
    example_queries = [
        "Show me allele frequencies for Vgsc in Kenya",
        "How have kdr mutations changed over time in Ghana?",
        "Compare Fst between populations on chromosome 2L by country",
        "What samples do we have from Burkina Faso?",
    ]
    print("Example queries:")
    for eq in example_queries:
        print(f"  - {eq}")
    print()
    
    while True:
        try:
            query = input("Your question: ").strip()
        except (EOFError, KeyboardInterrupt):
            break
            
        if query.lower() in ('quit', 'exit', 'q'):
            break
        if not query:
            continue
            
        result = parse_query(query)
        
        print(f"\n  Intent:      {result.intent}")
        print(f"  Confidence:  {result.confidence:.0%}")
        print(f"  Entities:    {result.entities}")
        print(f"  Explanation: {result.explanation}")
        print(f"\n  Generated code:")
        for line in result.api_call.split('\n'):
            print(f"    {line}")
        print()

# Uncomment to run interactively:
# interactive_demo()

# Non-interactive demonstration
print("Interactive demo available — uncomment interactive_demo() to try.\n")
print("Example session:")
print('  Question: "Show me resistance mutations in Cameroon"')
result = parse_query("Show me resistance mutations in Cameroon")
print(f"  Intent:      {result.intent}")
print(f"  Confidence:  {result.confidence:.0%}")
print(f"  Entities:    {result.entities}")
print(f"  Explanation: {result.explanation}")
print(f"\n  Generated code:")
for line in result.api_call.split('\n'):
    print(f"    {line}")

Interactive demo available — uncomment interactive_demo() to try.

Example session:
  Question: "Show me resistance mutations in Cameroon"
  Intent:      plot_frequencies_heatmap
  Confidence:  60%
  Entities:    {'country': 'cameroon', 'area': 'CM'}
  Explanation: Plots allele frequency heatmap for Vgsc gene in Cameroon

  Generated code:
    ag3.plot_frequencies_heatmap(
        transcript="AGAP004707-RD",
        sample_query="country == 'Cameroon'",
        min_cohort_size=10
    )


## 9. Summary & Next Steps

Performance summary, demonstrated capabilities, known limitations, and proposed GSoC enhancements.

In [10]:
# Summary & Conclusion

print("""
══════════════════════════════════════════════════════════════════════════════
  NLP INTERFACE PoC — RESULTS SUMMARY
══════════════════════════════════════════════════════════════════════════════

  PERFORMANCE (10 test queries + 3 edge cases)
  ─────────────────────────────────────────────
  Queries resolved:        10/10 (100%)
  Average confidence:      84%
  Total entities extracted: 29
  Edge cases handled:      3/3 (graceful degradation)

  CAPABILITIES DEMONSTRATED
  ─────────────────────────
  - Gene name → transcript ID mapping    (vgsc → AGAP004707-RD)
  - Country → area code resolution       (kenya → KE)
  - Temporal query detection             ("over time" → time_series)
  - Population comparison parsing        ("compare...by country" → Fst)
  - Species name normalization           (An. arabiensis → arabiensis)
  - Multi-species extraction             (gambiae + coluzzii → plot_snps_dxy)
  - Region expansion                     (East Africa → KE, TZ, UG, ET, MZ, MW)
  - Annotation preference extraction     ("lower triangle" → annotation)
  - Multi-entity extraction              (gene + country in one query)
  - Executable Python code generation    (valid malariagen_data calls)
  - Graceful out-of-scope handling       (weather, drugs, GWAS → "unknown")

  KNOWN LIMITATIONS (PoC scope)
  ─────────────────────────────
  - Keyword-based intent — no paraphrase handling
  - No disambiguation for ambiguous queries
  - No multi-step query chaining (e.g., "compute Fst then plot it")
  - No parameter validation against live API signatures
  - Fixed entity dictionaries — no learning from new data

  PROPOSED GSoC ENHANCEMENTS
  ──────────────────────────
  1. Replace keyword classifier with fine-tuned LLM (LoRA on domain
     examples, leveraging llama-task-agent architecture)
  2. Use RAG over API docstrings + training course notebooks for
     grounding (leveraging complaint-intelligence-system RAG pipeline)
  3. Validate generated calls against @_check_types decorator and
     Annotated type aliases at runtime
  4. Build evaluation suite: 50+ queries with expected API calls
  5. Add conversational context for multi-turn analysis sessions

  RELEVANT PRIOR WORK BY AUTHOR
  ─────────────────────────────
  - llama-task-agent    — LLaMA-3.1-8B fine-tuned with LoRA for
                          NL → structured task execution (100% format
                          compliance). Architecture directly applicable.
  - biodiversity-       — SciBERT on genomics literature (236 domain
    publication-analyzer  terms, 99.5% F1). Domain vocabulary transferable.
  - complaint-          — RAG pipeline with FAISS + Gemini over 12M+
    intelligence-system   records. Retrieval architecture applicable.

  Author: Aswani Sahoo (@AswaniSahoo)
══════════════════════════════════════════════════════════════════════════════
""")


══════════════════════════════════════════════════════════════════════════════
  NLP INTERFACE PoC — RESULTS SUMMARY
══════════════════════════════════════════════════════════════════════════════

  PERFORMANCE (10 test queries + 3 edge cases)
  ─────────────────────────────────────────────
  Queries resolved:        10/10 (100%)
  Average confidence:      84%
  Total entities extracted: 29
  Edge cases handled:      3/3 (graceful degradation)

  CAPABILITIES DEMONSTRATED
  ─────────────────────────
  - Gene name → transcript ID mapping    (vgsc → AGAP004707-RD)
  - Country → area code resolution       (kenya → KE)
  - Temporal query detection             ("over time" → time_series)
  - Population comparison parsing        ("compare...by country" → Fst)
  - Species name normalization           (An. arabiensis → arabiensis)
  - Multi-species extraction             (gambiae + coluzzii → plot_snps_dxy)
  - Region expansion                     (East Africa → KE, TZ, UG, ET, MZ, MW)
  - An